<a href="https://colab.research.google.com/github/JamesEckhartJr/Quantum-Computing-Projects/blob/main/Quantum_Teleportation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import requests
from collections import Counter

DEVICE = 'theq-ae31b1'          # change if needed
URL = f'https://{DEVICE}.quokkacomputing.com/qsim/qasm'
SHOTS = 8192

# Teleport the |0> state
qasm = """
OPENQASM 2.0;
include "qelib1.inc";
qreg q[3];
creg c[3];

// State to teleport: |0> on qubit 0 (Alice)
// (leave as |0> — no gates needed)

// Create Bell pair between qubit 1 (Alice) and qubit 2 (Bob)
h q[1];
cx q[1], q[2];

// Alice's Bell measurement
cx q[0], q[1];
h q[0];

// Measure all
measure q[0] -> c[0];
measure q[1] -> c[1];
measure q[2] -> c[2];
"""

print("Running 3-qubit Quantum Teleportation on Quokka...\n")
payload = {"script": qasm.strip(), "count": SHOTS}
r = requests.post(URL, json=payload, timeout=60)
shots = r.json()["result"]["c"]

# Apply classical corrections and count successful teleportations
success = 0
correction_counts = Counter()

for bits in shots:
    c0, c1, c2 = bits          # c0, c1 = Alice's Bell results; c2 = Bob's raw bit

    # Record which correction was needed
    corr = f"{c0}{c1}"
    correction_counts[corr] += 1

    # Apply corrections to Bob's qubit
    if c1 == 1:                # X correction
        c2 = 1 - c2
    if c0 == 1:                # Z correction (no effect on Z measurement)
        pass

    # After corrections, Bob should have |0> (measured as 0)
    if c2 == 0:
        success += 1

fidelity = success / len(shots)

print("=" * 60)
print("QUANTUM TELEPORTATION RESULTS")
print("=" * 60)
print(f"State teleported     : |0⟩")
print(f"Shots                : {len(shots)}")
print(f"Successful outcomes  : {success}")
print(f"Fidelity             : {fidelity:.4f}  ({fidelity*100:.2f}%)")
print()
print("Bell measurement outcomes (Alice) and required corrections:")
for corr in ["00", "01", "10", "11"]:
    count = correction_counts.get(corr, 0)
    pauli = {"00": "I", "01": "X", "10": "Z", "11": "Y"}[corr]
    print(f"  {corr} → {pauli:2s} correction : {count:5d} shots  ({count/len(shots)*100:5.1f}%)")
print("=" * 60)
print("""
EXPLANATION:
- A perfect teleportation of |0⟩ should yield fidelity ≈ 1.0 on a noise-free emulator.
- Alice’s two measurement bits determine which Pauli correction Bob must apply.
- After the classical corrections are applied in post-processing, Bob’s qubit
  matches the original state.
- The four Bell outcomes should each appear ~25% of the time.
- This demonstrates the full teleportation protocol (entanglement + Bell
  measurement + classical communication + correction) on the da Vinci/Quokka stack.
""")

Running 3-qubit Quantum Teleportation on Quokka...

QUANTUM TELEPORTATION RESULTS
State teleported     : |0⟩
Shots                : 8192
Successful outcomes  : 8192
Fidelity             : 1.0000  (100.00%)

Bell measurement outcomes (Alice) and required corrections:
  00 → I  correction :  1979 shots  ( 24.2%)
  01 → X  correction :  2048 shots  ( 25.0%)
  10 → Z  correction :  2101 shots  ( 25.6%)
  11 → Y  correction :  2064 shots  ( 25.2%)

EXPLANATION:
- A perfect teleportation of |0⟩ should yield fidelity ≈ 1.0 on a noise-free emulator.
- Alice’s two measurement bits determine which Pauli correction Bob must apply.
- After the classical corrections are applied in post-processing, Bob’s qubit
  matches the original state.
- The four Bell outcomes should each appear ~25% of the time.
- This demonstrates the full teleportation protocol (entanglement + Bell
  measurement + classical communication + correction) on the da Vinci/Quokka stack.

